# Feature Engineering v2 — 滑动窗口重构
**date**: 2026-04-14

改进点：
1. **区间滑窗提取** — 两个训练窗口 + 一个 618 预测窗口
2. **convert_discount_rate** — 将满减/折扣率统一转为实付比例浮点数
3. **User-Merchant 交互特征** — 统计每个用户在各商户的历史点击 & 购买次数


In [16]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')


Libraries loaded.


In [17]:
df = pd.read_csv('../data/raw/online_data.csv')
print(f'原始数据 shape: {df.shape}')
print(df.head(3).to_string())


原始数据 shape: (11429826, 7)
    User_id  Merchant_id  Action  Coupon_id Discount_rate  Date_received        Date
0  13740231        18907       2  100017492        500:50     20160513.0         NaN
1  13740231        34805       1        NaN           NaN            NaN  20160321.0
2  14336199        18907       0        NaN           NaN            NaN  20160618.0


## 1. 数据预处理

In [18]:
# 列名统一小写
df.columns = df.columns.str.lower()

# ── 日期转换 ──────────────────────────────────────────────────────────────────
def parse_date(series):
    """将 20160513.0 (float) 转为 pd.Timestamp"""
    unique_vals = series.dropna().unique()
    date_map = {}
    for v in unique_vals:
        try:
            date_map[v] = pd.to_datetime(str(int(v)), format='%Y%m%d', errors='coerce')
        except Exception:
            date_map[v] = pd.NaT
    return series.map(date_map)

df['date']          = parse_date(df['date'])
df['date_received'] = parse_date(df['date_received'])

# ── ID 字段转字符串 ────────────────────────────────────────────────────────────
df['user_id']     = df['user_id'].astype(str)
df['merchant_id'] = df['merchant_id'].astype(str)

# coupon_id：NaN → 字符串 'nan'（避免 groupby 丢失）
df['coupon_id']     = df['coupon_id'].fillna('').replace('', 'NO_COUPON').astype(str)
df['discount_rate'] = df['discount_rate'].fillna('').replace('', None)

print(f'日期范围  date: {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'date_received: {df["date_received"].min().date()} ~ {df["date_received"].max().date()}')
print(f'唯一用户数: {df["user_id"].nunique():,}')


日期范围  date: 2016-01-01 ~ 2016-06-30
date_received: 2016-01-01 ~ 2016-06-15
唯一用户数: 762,858


## 2. convert_discount_rate — 优惠券折扣率转换

In [19]:
FIXED_DISCOUNT_CONST = 0.001   # 固定金额券（无法量化折比时）赋极低常数

def convert_discount_rate(rate):
    """
    将 Discount_rate 字段转为实付比例浮点数 (0, 1]
    -------------------------------------------------------
    '200:50'  → 1 - 50/200  = 0.75   (满200减50，实付75%)
    '0.8'     → 0.8                  (直接折扣率，实付80%)
    'fixed'   → FIXED_DISCOUNT_CONST (固定金额券)
    NaN / ''  → np.nan
    """
    if pd.isna(rate) or str(rate).strip() == '':
        return np.nan
    s = str(rate).strip().lower()

    if ':' in s:
        try:
            parts     = s.split(':')
            threshold = float(parts[0])
            discount  = float(parts[1])
            if threshold <= 0:
                return np.nan
            return round(1.0 - discount / threshold, 6)   # 实付比例
        except Exception:
            return FIXED_DISCOUNT_CONST

    if s in ('fixed', 'nan', 'none', ''):
        return FIXED_DISCOUNT_CONST

    try:
        val = float(s)
        if 0 < val <= 1:
            return val           # 已经是折扣率（如 0.8）
        elif val > 1:
            return round(val / 100.0, 6)   # 百分比形式（如 80 → 0.80）
        else:
            return FIXED_DISCOUNT_CONST
    except Exception:
        return FIXED_DISCOUNT_CONST

# 生成数值化折扣率列
df['discount_rate_num'] = df['discount_rate'].apply(convert_discount_rate)

# 快速验证
sample = (
    df[df['discount_rate'].notna()][['discount_rate', 'discount_rate_num']]
    .drop_duplicates()
    .head(10)
)
print('折扣率转换示例:')
print(sample.to_string(index=False))


折扣率转换示例:
discount_rate  discount_rate_num
       500:50           0.900000
       150:50           0.666667
         50:5           0.900000
         30:1           0.966667
       300:50           0.833333
       300:30           0.900000
       800:50           0.937500
     1000:100           0.900000
         10:5           0.500000
       200:30           0.850000


## 3. 行为标志

In [20]:
# Action: 0=点击  1=购买（无券/有券）  2=领券未购买
df['is_click']            = (df['action'] == 0).astype(int)
df['is_purchase']         = (df['action'] == 1).astype(int)
df['is_coupon_received']  = (df['action'] == 2).astype(int)

# 有券购买：Action==1 且 coupon_id 非空
df['is_coupon_purchase']     = (
    (df['action'] == 1) & (df['coupon_id'] != 'NO_COUPON')
).astype(int)
df['is_no_coupon_purchase']  = (
    (df['action'] == 1) & (df['coupon_id'] == 'NO_COUPON')
).astype(int)
# 领了券但未购买
df['is_coupon_not_used'] = (
    (df['action'] == 2) & df['date'].isna()
).astype(int)

for col in ['is_click', 'is_purchase', 'is_coupon_received',
            'is_coupon_purchase', 'is_no_coupon_purchase', 'is_coupon_not_used']:
    print(f'  {col}: {df[col].sum():>9,}')


  is_click: 9,401,780
  is_purchase: 1,372,148
  is_coupon_received:   655,898
  is_coupon_purchase:   216,459
  is_no_coupon_purchase: 1,155,689
  is_coupon_not_used:   655,898


## 4. 特征提取函数

In [21]:
def extract_user_basic_features(feat_data):
    """用户基础统计特征 - 修复版"""
    # 使用字典聚合，避免元组传参触发 TypeError
    basic = feat_data.groupby('user_id').agg({
        'is_purchase': 'sum',
        'is_click': 'sum',
        'is_coupon_received': 'sum',
        'is_coupon_purchase': 'sum',
        'is_no_coupon_purchase': 'sum',
        'is_coupon_not_used': 'sum',
        'merchant_id': 'nunique',
        'coupon_id': 'nunique'
    }).reset_index()

    # 重命名列名
    basic.columns = [
        'user_id', 'total_purchases', 'total_clicks', 'total_coupon_rcvd',
        'coupon_purchases', 'no_coupon_purchases', 'coupons_not_used',
        'merchant_count', 'coupon_types_count'
    ]

    # 逻辑计算
    r = basic
    r['coupon_use_rate'] = r['coupon_purchases'] / r['total_coupon_rcvd'].replace(0, 1)
    r['coupon_abandon_rate'] = r['coupons_not_used'] / r['total_coupon_rcvd'].replace(0, 1)
    r['coupon_purchase_ratio'] = r['coupon_purchases'] / r['total_purchases'].replace(0, 1)
    r['click_to_buy_rate'] = r['total_purchases'] / r['total_clicks'].replace(0, 1)
    r['avg_purchase_per_merchant'] = r['total_purchases'] / r['merchant_count'].replace(0, 1)
    
    return basic

In [22]:
def extract_user_merchant_features(feat_data):
    """User-Merchant 交互特征 """
    click_data    = feat_data[feat_data['is_click']    == 1]
    purchase_data = feat_data[feat_data['is_purchase'] == 1]

    um_clicks = click_data.groupby(['user_id', 'merchant_id']).size().reset_index(name='um_click_count')
    um_purchases = purchase_data.groupby(['user_id', 'merchant_id']).size().reset_index(name='um_purchase_count')
    
    um = um_clicks.merge(um_purchases, on=['user_id', 'merchant_id'], how='outer').fillna(0)

    # 聚合到用户级别
    um_agg = um.groupby('user_id').agg({
        'merchant_id': 'count',
        'um_click_count': ['max', 'mean', 'sum'],
        'um_purchase_count': ['max', 'mean', 'sum']
    })
    
    # 展平多级索引
    um_agg.columns = [
        'um_total_pairs', 'um_max_clicks', 'um_avg_clicks', 
        'um_total_clicks', 'um_max_purchases', 'um_avg_purchases', 'um_total_um_purch'
    ]
    um_agg = um_agg.reset_index()

    buy_pairs = um[um['um_purchase_count'] > 0].groupby('user_id').size().reset_index(name='um_purchase_pairs')
    um_agg = um_agg.merge(buy_pairs, on='user_id', how='left').fillna(0)
    
    um_agg['um_purchase_penetration'] = um_agg['um_purchase_pairs'] / um_agg['um_total_pairs'].replace(0, 1)
    return um_agg

In [23]:
def extract_time_window_features(feat_data, feat_end, windows=(7, 14, 30)):
    """滑动窗口特征"""
    base = pd.DataFrame({'user_id': feat_data['user_id'].unique()})

    for w in windows:
        start = feat_end - pd.Timedelta(days=w)
        w_data = feat_data[feat_data['date'] >= start]
        
        # 统一使用字典传参
        w_stats = w_data.groupby('user_id').agg({
            'is_purchase': 'sum',
            'is_click': 'sum',
            'is_coupon_received': 'sum'
        }).reset_index()
        
        w_stats.columns = ['user_id', f'purchases_{w}d', f'clicks_{w}d', f'coupons_{w}d']
        base = base.merge(w_stats, on='user_id', how='left')

    fill_cols = [c for c in base.columns if c != 'user_id']
    base[fill_cols] = base[fill_cols].fillna(0)
    return base

In [24]:
def extract_rfm_features(feat_data, feat_end):
    """RFM 特征 - 修复版"""
    purch = feat_data[feat_data['is_purchase'] == 1].copy()
    if len(purch) == 0:
        return pd.DataFrame({'user_id': feat_data['user_id'].unique()})

    rfm_raw = purch.groupby('user_id').agg({
        'date': ['min', 'max', 'count']
    })
    
    rfm_raw.columns = ['first_purchase_date', 'last_purchase_date', 'rfm_frequency']
    rfm_raw = rfm_raw.reset_index()

    rfm_raw['recency']  = (feat_end - rfm_raw['last_purchase_date']).dt.days
    rfm_raw['lifetime'] = (rfm_raw['last_purchase_date'] - rfm_raw['first_purchase_date']).dt.days
    rfm_raw['purchase_frequency_daily'] = rfm_raw['rfm_frequency'] / rfm_raw['lifetime'].replace(0, 1)
    
    return rfm_raw[['user_id', 'recency', 'lifetime', 'rfm_frequency', 'purchase_frequency_daily']]

In [25]:
def extract_coupon_features(feat_data):
    """优惠券特征 - 修复版"""
    cp = feat_data[feat_data['is_coupon_purchase'] == 1].copy()
    if len(cp) == 0:
        return pd.DataFrame({'user_id': feat_data['user_id'].unique()})

    disc_stats = cp.groupby('user_id').agg({
        'discount_rate_num': ['mean', 'median', 'std', 'min', 'max']
    })
    disc_stats.columns = ['avg_discount', 'median_discount', 'std_discount', 'min_discount', 'max_discount']
    disc_stats = disc_stats.reset_index()

    cp_used = feat_data[feat_data['is_coupon_purchase'].eq(1) & feat_data['date_received'].notna()].copy()
    if len(cp_used) > 0:
        cp_used['days_to_use'] = (cp_used['date'] - cp_used['date_received']).dt.days
        days_stats = cp_used.groupby('user_id').agg({
            'days_to_use': ['mean', 'max']
        })
        days_stats.columns = ['avg_days_to_use', 'max_days_to_use']
        days_stats = days_stats.reset_index()
        disc_stats = disc_stats.merge(days_stats, on='user_id', how='left')

    return disc_stats

In [26]:
def extract_time_preference_features(feat_data):
    """时间偏好特征 - 修复版"""
    purch = feat_data[feat_data['is_purchase'] == 1].copy()
    if len(purch) == 0:
        return pd.DataFrame({'user_id': feat_data['user_id'].unique()})

    purch['weekday'] = purch['date'].dt.weekday
    wkd = purch[purch['weekday'] < 5].groupby('user_id').size().reset_index(name='weekday_purchases')
    wke = purch[purch['weekday'] >= 5].groupby('user_id').size().reset_index(name='weekend_purchases')

    base = pd.DataFrame({'user_id': feat_data['user_id'].unique()})
    base = base.merge(wkd, on='user_id', how='left')
    base = base.merge(wke, on='user_id', how='left')
    base[['weekday_purchases', 'weekend_purchases']] = base[['weekday_purchases', 'weekend_purchases']].fillna(0)
    
    base['weekend_purchase_ratio'] = base['weekend_purchases'] / (base['weekday_purchases'] + base['weekend_purchases']).replace(0, 1)
    return base

In [27]:
def build_features(df, feat_start, feat_end):
    """
    主特征提取：从 df 中提取 [feat_start, feat_end] 内的所有用户特征。
    date 或 date_received 落在区间内的记录均纳入统计。
    """
    feat_data = df[
        ((df['date']          >= feat_start) & (df['date']          <= feat_end)) |
        ((df['date_received'] >= feat_start) & (df['date_received'] <= feat_end))
    ].copy()

    active_users = pd.DataFrame({'user_id': feat_data['user_id'].unique()})

    feats = [
        extract_user_basic_features(feat_data),
        extract_user_merchant_features(feat_data),
        extract_time_window_features(feat_data, feat_end),
        extract_rfm_features(feat_data, feat_end),
        extract_coupon_features(feat_data),
        extract_time_preference_features(feat_data),
    ]

    result = active_users
    for f in feats:
        result = result.merge(f, on='user_id', how='left')

    # 数值列填 0
    num_cols = result.select_dtypes(include=np.number).columns
    result[num_cols] = result[num_cols].fillna(0)
    return result


In [28]:
def build_dataset(df, feat_start, feat_end, label_start=None, label_end=None):
    """
    构建完整数据集（特征 + 标签）。
    label_start / label_end 为 None 时生成纯特征集（测试集）。
    """
    print(f'  特征区间: {feat_start.date()} ~ {feat_end.date()}')
    features = build_features(df, feat_start, feat_end)

    if label_start is not None and label_end is not None:
        print(f'  标签区间: {label_start.date()} ~ {label_end.date()}')
        buyers = df[
            (df['date'] >= label_start) &
            (df['date'] <= label_end)   &
            df['date'].notna()
        ]['user_id'].unique()

        features['label'] = features['user_id'].isin(buyers).astype(int)
        pos = features['label'].mean() * 100
        print(f'  样本数: {len(features):,}   正样本率: {pos:.2f}%')

    return features


## 5. 滑动窗口构建

In [29]:
# ── 滑动窗口参数 ──────────────────────────────────────────────────────────────
W1_FEAT_START  = pd.Timestamp('2016-01-01')
W1_FEAT_END    = pd.Timestamp('2016-04-13')
W1_LABEL_START = pd.Timestamp('2016-04-14')
W1_LABEL_END   = pd.Timestamp('2016-05-14')

W2_FEAT_START  = pd.Timestamp('2016-02-01')
W2_FEAT_END    = pd.Timestamp('2016-05-14')
W2_LABEL_START = pd.Timestamp('2016-05-15')
W2_LABEL_END   = pd.Timestamp('2016-06-15')

# 618 测试集：特征区间结束后预测未来 15 天购买行为
TEST_FEAT_START  = pd.Timestamp('2016-03-15')
TEST_FEAT_END    = pd.Timestamp('2016-06-15')
TEST_LABEL_START = pd.Timestamp('2016-06-16')
TEST_LABEL_END   = pd.Timestamp('2016-06-30')

print('滑动窗口配置:')
print(f'  训练集1: 特征 [{W1_FEAT_START.date()} ~ {W1_FEAT_END.date()}]')
print(f'           标签 [{W1_LABEL_START.date()} ~ {W1_LABEL_END.date()}]')
print(f'  训练集2: 特征 [{W2_FEAT_START.date()} ~ {W2_FEAT_END.date()}]')
print(f'           标签 [{W2_LABEL_START.date()} ~ {W2_LABEL_END.date()}]')
print(f'  测试集:  特征 [{TEST_FEAT_START.date()} ~ {TEST_FEAT_END.date()}]')
print(f'           标签 [{TEST_LABEL_START.date()} ~ {TEST_LABEL_END.date()}]')


滑动窗口配置:
  训练集1: 特征 [2016-01-01 ~ 2016-04-13]
           标签 [2016-04-14 ~ 2016-05-14]
  训练集2: 特征 [2016-02-01 ~ 2016-05-14]
           标签 [2016-05-15 ~ 2016-06-15]
  测试集:  特征 [2016-03-15 ~ 2016-06-15]
           标签 [2016-06-16 ~ 2016-06-30]


In [30]:

print('构建训练集 1 ...')
train_w1 = build_dataset(df, W1_FEAT_START, W1_FEAT_END, W1_LABEL_START, W1_LABEL_END)
print(f'  shape: {train_w1.shape}')

构建训练集 1 ...
  特征区间: 2016-01-01 ~ 2016-04-13
  标签区间: 2016-04-14 ~ 2016-05-14
  样本数: 616,543   正样本率: 46.38%
  shape: (616543, 47)


In [31]:
print('构建训练集 2 ...')
train_w2 = build_dataset(df, W2_FEAT_START, W2_FEAT_END, W2_LABEL_START, W2_LABEL_END)
print(f'  shape: {train_w2.shape}')


构建训练集 2 ...
  特征区间: 2016-02-01 ~ 2016-05-14
  标签区间: 2016-05-15 ~ 2016-06-15
  样本数: 626,995   正样本率: 47.80%
  shape: (626995, 47)


In [32]:
print('构建测试集 (618 促销期) ...')
test_618 = build_dataset(df, TEST_FEAT_START, TEST_FEAT_END, TEST_LABEL_START, TEST_LABEL_END)
print(f'  shape: {test_618.shape}')


构建测试集 (618 促销期) ...
  特征区间: 2016-03-15 ~ 2016-06-15
  标签区间: 2016-06-16 ~ 2016-06-30
  样本数: 638,373   正样本率: 32.62%
  shape: (638373, 47)


## 6. 合并 & 保存

In [33]:
import os
os.makedirs('../data/features', exist_ok=True)

# 合并两个训练窗口
train_combined = pd.concat([train_w1, train_w2], ignore_index=True)
print(f'合并训练集 shape: {train_combined.shape}')
print(f'正样本率: {train_combined["label"].mean()*100:.2f}%')

# 特征列
feature_cols = [c for c in train_combined.columns if c not in ('user_id', 'label')]
print(f'特征数量: {len(feature_cols)}')

# 保存
train_combined.to_csv('../data/features/train_combined.csv', index=False)
test_618.to_csv('../data/features/test_618.csv', index=False)

# 特征列表
with open('../data/features/feature_list_v2.txt', 'w', encoding='utf-8') as f:
    f.write(f'# Feature List v2 (Sliding Window)\n')
    f.write(f'# Generated: {datetime.now()}\n')
    f.write(f'# Feature count: {len(feature_cols)}\n\n')
    for col in feature_cols:
        f.write(col + '\n')

print('\n已保存:')
print('  ../data/features/train_combined.csv')
print('  ../data/features/test_618.csv')
print('  ../data/features/feature_list_v2.txt')


合并训练集 shape: (1243538, 47)
正样本率: 47.09%
特征数量: 45

已保存:
  ../data/features/train_combined.csv
  ../data/features/test_618.csv
  ../data/features/feature_list_v2.txt
